# 🌿 Project 17 — Monthly Greenest Image

Excellent! This project introduces one of the most useful image compositing techniques in Google Earth Engine.

Until now, you've worked with:

- A **single satellite image**
- Two images for **change detection**
- Many images for **time series analysis**

Now you'll combine **all images from one month** into a single image that keeps the **greenest vegetation** observed during that month.

This technique is widely used in professional remote sensing because it reduces clouds and produces a cleaner representation of vegetation.

---

# 🎯 Goal

Create a **monthly vegetation composite** that keeps the greenest vegetation observed during one month.

Instead of selecting one satellite image, you'll combine many images into a single, cloud-reduced composite.

---

# 🌍 Why is this useful in Remote Sensing?

Monthly composites are widely used in:

- 🌾 Crop monitoring
- 🌿 Seasonal vegetation studies
- 🌍 Climate analysis
- 🌳 Forest monitoring
- 🌱 Drought assessment

Using many images helps reduce the effects of:

- ☁ Clouds
- 🌫 Haze
- 🚜 Temporary disturbances

---

# ⭐ New Skill: Image Composites

Previously your workflow looked like this:

```text
Image Collection
       ↓
.first()
       ↓
One Image
```

Today it becomes:

```text
Image Collection
       ↓
Calculate NDVI
       ↓
Many NDVI Images
       ↓
qualityMosaic()
       ↓
One Greenest Image
```

Instead of selecting the **first image**, we create a **new image** using information from **all images**.

---

# Before We Start

In **Project 16**, you learned:

```javascript
collection.map(calculateNDVI)
```

This produced:

```text
NDVI Image Collection
```

Today we'll use that same idea again.

This project builds directly on what you learned previously.

---

# What is a Composite?

Imagine you have four satellite images.

```text
Week 1
☁ Cloudy

Week 2
🌿 Green

Week 3
☁ Cloudy

Week 4
🌿 Very Green
```

Instead of choosing only one image,

Earth Engine looks at **every pixel** across all images and keeps the **best one**.

```text
Image 1
      ↓

Image 2
      ↓

Image 3
      ↓

Image 4
      ↓

Best Pixel
      ↓

Composite Image
```

---

# What Does "Greenest" Mean?

Remember NDVI.

Higher NDVI means:

- Healthier vegetation
- More active plant growth

Example:

| Image | NDVI |
|-------|------|
| 1 | 0.42 |
| 2 | 0.71 |
| 3 | 0.58 |
| 4 | 0.81 |

Earth Engine keeps:

```text
0.81
```

because it is the **highest NDVI**.

This process is repeated for **every pixel** in the image.

---

# Step 1 — Define the Study Area

We'll use Germany again.

```javascript
var countryArea = ee.Geometry.Polygon([
  [
    [5.87,47.27],
    [15.04,47.27],
    [15.04,55.06],
    [5.87,55.06],
    [5.87,47.27]
  ]
]);
```

---

# Step 2 — Load the Sentinel-2 Collection

This time we'll use one month.

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-06-01','2024-06-30');
```

Notice:

We still do **not** use:

```javascript
.first()
```

Why?

Because we need **all images** from the month.

---

# Step 3 — Create an NDVI Function

We'll modify our previous function slightly.

```javascript
function addNDVI(image){

  var ndvi = image
    .normalizedDifference(['B8','B4'])
    .rename('NDVI');

  return image.addBands(ndvi);

}
```

---

## Why don't we return only NDVI?

In **Project 16**, we returned:

```javascript
return ndvi;
```

because we only wanted to create an NDVI chart.

Today we want to display a **true-color satellite image**.

So we keep:

- Blue
- Green
- Red
- NIR
- All original bands

and simply add a new **NDVI** band.

Workflow:

```text
Original Image
(B2 B3 B4 B8 ...)
        +
      NDVI
        ↓
New Image
(B2 B3 B4 B8 NDVI)
```

This is why we use:

```javascript
image.addBands(ndvi)
```

---

# Step 4 — Apply the Function

```javascript
var ndviCollection = collection.map(addNDVI);
```

Exactly like Project 16.

Now every image contains an additional NDVI band.

---

# Step 5 — Create the Greenest Composite ⭐

Today's new function:

```javascript
var greenest = ndviCollection.qualityMosaic('NDVI');
```

This is one of the most useful functions in Earth Engine.

---

## What is `qualityMosaic()`?

Think of it like a competition.

Each pixel competes with the same pixel from every image.

```text
Image 1
NDVI = 0.41

Image 2
NDVI = 0.67

Image 3
NDVI = 0.54

Image 4
NDVI = 0.82
```

Winner:

```text
0.82
```

Earth Engine copies **all bands** from the winning image for that pixel.

This is why we needed to keep the original bands using:

```javascript
image.addBands(ndvi)
```

---

# Step 6 — Create Visualization

```javascript
var trueColor = {
  bands:['B4','B3','B2'],
  min:0,
  max:3000
};
```

We'll display the composite as a true-color image.

---

# Step 7 — Display the Composite

```javascript
Map.centerObject(countryArea,5);

Map.addLayer(
  greenest,
  trueColor,
  'Monthly Greenest Composite'
);
```

Now you'll see a cloud-reduced image where vegetation appears at its healthiest during June.

---

# Complete Code

```javascript
// 1. Study Area
var countryArea = ee.Geometry.Polygon([
  [
    [5.87,47.27],
    [15.04,47.27],
    [15.04,55.06],
    [5.87,55.06],
    [5.87,47.27]
  ]
]);

// 2. Load Collection
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-06-01','2024-06-30');

// 3. Add NDVI Band
function addNDVI(image){

  var ndvi = image
    .normalizedDifference(['B8','B4'])
    .rename('NDVI');

  return image.addBands(ndvi);

}

// 4. Apply Function
var ndviCollection = collection.map(addNDVI);

// 5. Greenest Composite
var greenest = ndviCollection.qualityMosaic('NDVI');

// 6. Visualization
var trueColor = {
  bands:['B4','B3','B2'],
  min:0,
  max:3000
};

// 7. Display
Map.centerObject(countryArea,5);

Map.addLayer(
  greenest,
  trueColor,
  'Monthly Greenest Composite'
);
```

---

# Things to Remember

## 1. `addBands()`

Previously:

```text
Image
   ↓
NDVI
```

Now:

```text
Image
    +
NDVI
    ↓
Image with Extra Band
```

`addBands()` keeps the original image and adds a new band.

---

## 2. `qualityMosaic()`

```javascript
qualityMosaic('NDVI')
```

For every pixel, Earth Engine:

- Looks at all images
- Finds the highest NDVI value
- Copies **all bands** from that image

Workflow:

```text
Many Images
      ↓
Highest NDVI Pixel
      ↓
Composite Image
```

---

## 3. Image Collection vs Composite

Remember the difference.

### Image Collection

```text
Many Images
```

### Composite Image

```text
Many Images
      ↓
qualityMosaic()
      ↓
One Image
```

This is one of the most important concepts in Google Earth Engine.

---

## 4. Why Does `qualityMosaic()` Reduce Clouds?

Clouds usually have **low NDVI values** because they do not contain healthy vegetation.

Healthy vegetation generally has **high NDVI values**.

When Earth Engine selects the pixel with the **highest NDVI**, cloudy pixels are often ignored in favor of clear observations.

This produces a cleaner image without explicitly masking clouds.

---

## 5. Why Keep the Original Bands?

Suppose the winning pixel comes from Image 4.

Earth Engine copies:

```text
B2
B3
B4
B8
...
```

from Image 4,

not just the NDVI value.

That's why we must use:

```javascript
image.addBands(ndvi)
```

instead of returning only the NDVI image.

---

# Common Beginner Mistakes

❌ **Mistake 1:** Returning only the NDVI image.

```javascript
return ndvi;
```

Then `qualityMosaic()` creates an image containing only the NDVI band, so you cannot display it in true color.

Correct:

```javascript
return image.addBands(ndvi);
```

---

❌ **Mistake 2:** Using `.first()`.

```javascript
collection.first()
```

This keeps only one image, so there is nothing to combine into a composite.

---

❌ **Mistake 3:** Forgetting to add the NDVI band.

```javascript
qualityMosaic('NDVI')
```

requires every image in the collection to contain an NDVI band.

Without it, Earth Engine cannot determine which pixel is the "greenest."

---

❌ **Mistake 4:** Thinking `qualityMosaic()` averages images.

It does **not** calculate an average.

Instead, it selects the **best pixel** (highest NDVI) independently for every location.

---

# Your Practice (Project 17.1)

Create a monthly greenest composite for a study area of your choice.

## Requirements

1. Create a study area.
2. Load one month of Sentinel-2 images.
3. Write an `addNDVI()` function.
4. Use `.map()` to add the NDVI band to every image.
5. Create a greenest composite using:

```javascript
qualityMosaic('NDVI')
```

6. Display the composite in true color.

---

Before writing the code, answer these two questions.

## Question 1

Why do we use:

```javascript
image.addBands(ndvi)
```

instead of:

```javascript
return ndvi;
```

when creating a greenest composite?

---

## Question 2

What is the difference between:

```javascript
collection.first()
```

and

```javascript
collection.qualityMosaic('NDVI')
```

in terms of how the final image is created?

Send me your answers and your code, and I'll review both your understanding and your implementation exactly like a university professor.